# Revenue report — as it was handed over

A colleague wrote this notebook for the monthly board pack. The board pack needs four tables:

1. total revenue,
2. revenue by payment type,
3. revenue by product category,
4. revenue by customer state.

The data is a selected sample of the Olist marketplace (Brazil; money is in BRL), one CSV per table in `data/raw/`.
The cells under **The report** are your colleague's. They run without an error. Run them, read the numbers, and then
read the README: the report does not agree with itself.

*(The other notebook, `lecture.ipynb`, holds the lecture's queries for review. This lab does not need it.)*

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.max_rows", 400)   # show every row of a result: a census is never "the top few"

# Anchor to the project folder (DS1, Block 1), then stand there: every path below is from the project folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print("Working in:", Path.cwd())   # must print this project's folder

con = duckdb.connect()             # an in-memory database; it reads the files in data/raw/ directly

In [ ]:
# Every table as a view with its file's name, so a query can say FROM orders instead of the path.
# A view is a saved query (Lab 1): nothing is copied; the file in data/raw/ is read each time.
TABLES = ["orders", "order_items", "order_payments", "customers", "products", "product_category_name_translation"]
for t in TABLES:
    con.sql(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM 'data/raw/{t}.csv'")
print("views:", ", ".join(TABLES))

## The report

### 1. Total revenue

In [ ]:
total = con.sql("SELECT SUM(price) FROM order_items").fetchone()[0]
print(f"Total revenue: {total:,.2f} BRL")

### 2. Revenue by payment type

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW revenue_by_payment_type AS
    SELECT p.payment_type, ROUND(SUM(i.price), 2) AS revenue
    FROM order_items AS i
    JOIN order_payments AS p ON p.order_id = i.order_id
    GROUP BY p.payment_type
""")
display(con.sql("SELECT * FROM revenue_by_payment_type ORDER BY revenue DESC").df())
print(f"This table adds up to {con.sql('SELECT SUM(revenue) FROM revenue_by_payment_type').fetchone()[0]:,.2f} BRL")

### 3. Revenue by product category

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW revenue_by_category AS
    SELECT t.product_category_name_english AS category, ROUND(SUM(i.price), 2) AS revenue
    FROM order_items AS i
    JOIN products AS p ON p.product_id = i.product_id
    JOIN product_category_name_translation AS t ON t.product_category_name = p.product_category_name
    GROUP BY t.product_category_name_english
""")
display(con.sql("SELECT * FROM revenue_by_category ORDER BY revenue DESC").df())
print(f"This table adds up to {con.sql('SELECT SUM(revenue) FROM revenue_by_category').fetchone()[0]:,.2f} BRL")

### 4. Revenue by customer state

*Not written yet — your colleague ran out of time. It is section C below.*

---

# Your work starts here

**The first 20 minutes are AI-off** (the README says why). Work top to bottom. Paste what each inspection returns
into `DIAGNOSIS.md`, part 3, **before** you change anything.

Sections A to D are the core. **Start the note by minute 22**, finished or not: it needs five minutes. Section E is
for after the note, if there is time; F and G are the stretch.

## A. What did the join in section 2 do?

The lecture's habit: after every join, three counts. Here the left table is `order_items`, and one row of it is one
item, identified by `(order_id, order_item_id)`.

1. How many rows does `order_items` have?
2. How many rows does the join in section 2 return?
3. How many **distinct items** are in that result?

Say what you expect before you run it.

In [ ]:
# The three counts for section 2's join. One query or three: your choice.

Now find the orders that appear in the result **more times than they have items**. Pick one. How many rows does it
have in `order_payments`?

In [ ]:
# The orders that appear in the join more often than they have items; then one of them in order_payments.

Where does `payment_type` live? Run the cell (supplied), then answer in a comment: which table has it, what is one
row of that table, and which of the lecture's two patterns is *revenue by payment type*?

In [ ]:
con.sql("DESCRIBE order_payments").df()

In [ ]:
# One row of order_payments is one ...
# Revenue by payment type is pattern (a) / (b), because ...

## B. Revenue by payment type

**Write this query yourself, in the empty cell.** For each payment type: the revenue, meaning what customers paid
with it. One row per payment type, largest first. Save it as a view named `payment_type_revenue` with the columns
`payment_type` and `revenue`, then show it.

Before you run it, write in a comment how many rows you expect.

In [ ]:
# con.sql("""
#     CREATE OR REPLACE VIEW payment_type_revenue AS
#     ...                                  -- your query
# """)
# con.sql("SELECT * FROM payment_type_revenue ORDER BY revenue DESC").df()

## C. Revenue by customer state

The board pack's fourth table. For each customer state: **the number of orders, and the revenue** (what customers
paid). One row per state, largest revenue first. Save it as a view named `state_revenue` with the columns
`customer_state`, `orders`, and `revenue`, then show it.

`customer_state` is in `customers`; what customers paid is in `order_payments`; `orders` connects the two. Decide
which of the lecture's two patterns this is before you write it.

In [ ]:
# con.sql("""
#     CREATE OR REPLACE VIEW state_revenue AS
#     ...                                  -- your query
# """)
# con.sql("SELECT * FROM state_revenue ORDER BY revenue DESC").df()

Now the three counts for **your** join: rows going in, rows coming out, distinct orders coming out. Paste them into the comment, and into `DIAGNOSIS.md`.

In [ ]:
# The three counts for your join in C.
# rows going in: ...   rows coming out: ...   distinct orders coming out: ...

## D. The check: does every table add up to its own total?

Revenue is additive, so a breakdown of it must add up to its total, computed separately. Tables B and C break down
**what customers paid**, so their total is `SUM(payment_value)` over `order_payments` (the README's brief).

Print the payments total, what your table B adds up to, what your table C adds up to, and say for each whether it
matches. Then put C's `orders` column, added up, beside the number of orders that have a payment.

**How to compare money.** Two amounts match when they agree to the cent: `abs(a - b) < 0.005`. Never test money
with `==`. Sums of decimal amounts are computed in floating point and are rarely exact in their last digits: in this
file, the three items of order `181ff95f…`, 26.90 each, add up to `80.69999999999999`. Counts are whole numbers:
those must be exactly equal.

Then write the note. Section E is for after the note, if there is time.

In [ ]:
# The payments total; what B and C add up to, each against it (to the cent); C's orders against the orders with a payment.

## E. After the note, if there is time: the two totals

The report's total is a different measure: what the items sold for, `SUM(price)` over `order_items`. Print it beside
the payments total, and the difference between them. **Report the difference as a number. Do not explain it here**:
stretch G looks into it.

In [ ]:
# The items total, the payments total, and the difference between them.

## F. Stretch — the category table

Section 3 adds up to **less** than the items total. First find the items its join lost: an anti-join that lists each
category it lost, with its number of items and their revenue. Then write a labelled version of the table that keeps
every item, names the ones with no English category name as such, and adds up to the items total (to the cent:
`abs(a - b) < 0.005`).

Changing one word in section 3 is not the repair. The board pack has to say which items have no English name.

In [ ]:
# The anti-join: the categories section 3 lost, with their items and revenue.

In [ ]:
# The labelled table: every item kept, the untranslated ones named. Then what it adds up to.

## G. Stretch — is the difference freight?

`order_items` also has `freight_value`, the shipping charged per item. For each order that has both items and
payments, compare the items' price plus freight with what was paid. How many orders agree **to the cent**
(`abs(residual) < 0.005`), and how many do not? What about orders that are on only one side?

In [ ]:
# Per order: price + freight against what was paid. How many agree to the cent?